In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

### STEP 1 — Start Spark Session

In [0]:
spark = SparkSession.builder \
    .appName("SuperstoreETL") \
    .getOrCreate()
 
print("✓ Spark session started successfully\n")

✓ Spark session started successfully



In [0]:
spark

### STEP 2 — Read CSV

In [0]:
CSV_PATH      = "/Volumes/workspace/default/project_csv/store_sales/input/store_sales.csv"
CLEANED_PATH  = "/Volumes/workspace/default/project_csv/store_sales/output/superstore_cleaned"
CAT_PATH      = "/Volumes/workspace/default/project_csv/store_sales/output/category_sales"
REGION_PATH   = "/Volumes/workspace/default/project_csv/store_sales/output/region_sales"
SEGMENT_PATH  = "/Volumes/workspace/default/project_csv/store_sales/output/segment_sales"
SUBCAT_PATH   = "/Volumes/workspace/default/project_csv/store_sales/output/subcategory_sales"
SHIP_PATH     = "/Volumes/workspace/default/project_csv/store_sales/output/shipmode_sales"
DISCOUNT_PATH = "/Volumes/workspace/default/project_csv/store_sales/output/discount_analysis"
LOSS_PATH     = "/Volumes/workspace/default/project_csv/store_sales/output/loss_orders"

In [0]:
df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(CSV_PATH)

In [0]:
raw_count = df.count()
column = df.columns
print(f"✓ File loaded — {raw_count} rows, {len(column)} columns")
print(f"  Columns: {column}\n")

✓ File loaded — 9994 rows, 13 columns
  Columns: ['Ship Mode', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Category', 'Sub-Category', 'Sales', 'Quantity', 'Discount', 'Profit']



### STEP 3 — Inspect Schema


In [0]:
df.printSchema()
print("First 5 rows:")
df.limit(5).display(truncate=False)

root
 |-- Ship Mode: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)

First 5 rows:


Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.96,2,0.0,41.9136
Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.94,3,0.0,219.582
Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.62,2,0.0,6.8714
Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.031
Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.368,2,0.2,2.5164


### STEP 4 — Data Profiling

In [0]:
# Null counts per column
null_counts = df.select(
    [sum(
        when(isnull(c) | (col(c).cast("string") == ""), 1).otherwise(0)
    ).alias(c) for c in df.columns]
)
print("Null / empty counts per column:")
null_counts.display()

Null / empty counts per column:


Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Numeric statistics
print("Numeric column statistics (Sales, Quantity, Discount, Profit):")
df.select("Sales", "Quantity", "Discount", "Profit").describe().display()

Numeric column statistics (Sales, Quantity, Discount, Profit):


summary,Sales,Quantity,Discount,Profit
count,9994,9994,9994,9994
mean,229.8580008304935,3.789573744246548,0.15620272163298854,28.65689630778481
stddev,623.2451005086806,2.2251096911414003,0.20645196782571695,234.26010769095768
min,0.444,1,0.0,-6599.978
max,22638.48,14,0.8,8399.976


In [0]:
# Negative profit insight
loss_rows = df.filter(col("Profit") < 0).count()
print(f"⚠  Loss-making orders (Profit < 0) : {loss_rows} rows")
print(f"   Profit-making orders             : {raw_count - loss_rows} rows\n")

⚠  Loss-making orders (Profit < 0) : 1871 rows
   Profit-making orders             : 8123 rows



In [0]:
# Discount distribution
print("Discount value distribution:")
df.groupBy("Discount") \
  .count() \
  .orderBy("Discount") \
  .display(20, truncate=False)

Discount value distribution:


Discount,count
0.0,4798
0.1,94
0.15,52
0.2,3657
0.3,227
0.32,27
0.4,206
0.45,11
0.5,66
0.6,138


### STEP 5 — Handle Null Values

In [0]:
# Drop rows where critical columns are null
df = df.dropna(subset=["Sales", "Profit", "Category"])

In [0]:
# Fill remaining nulls
df = df.fillna({
    "Segment"      : "Unknown",
    "Region"       : "Unknown",
    "City"         : "Not Available",
    "State"        : "Not Available",
    "Ship Mode"    : "Standard Class",
    "Sub-Category" : "Unknown"
})

In [0]:
after_null_count = df.count()
print(f"✓ Rows before null handling : {raw_count}")
print(f"✓ Rows after null handling  : {after_null_count}")
print(f"  Dropped rows              : {raw_count - after_null_count}\n")

✓ Rows before null handling : 9994
✓ Rows after null handling  : 9994
  Dropped rows              : 0



### STEP 6 — Remove Duplicates

In [0]:
df = df.dropDuplicates()
after_dedup_count = df.count()
print(f"✓ Rows after deduplication : {after_dedup_count}")
print(f"  Duplicate rows removed   : {after_null_count - after_dedup_count}\n")

✓ Rows after deduplication : 9977
  Duplicate rows removed   : 17



### STEP 7 — Filter Invalid Data

In [0]:
# Keep only rows where Sales > 0 and Quantity > 0
df = df.filter((col("Sales") > 0) & (col("Quantity") > 0))

In [0]:
# Discount must be between 0 and 1 (it's a ratio)
df = df.filter((col("Discount") >= 0) & (col("Discount") <= 1))

In [0]:
after_filter_count = df.count()
print(f"✓ Rows after filtering : {after_filter_count}")
print(f"  Invalid rows removed : {after_dedup_count - after_filter_count}\n")

✓ Rows after filtering : 9977
  Invalid rows removed : 0



### STEP 8 — Drop Redundant Column

In [0]:
# 'Country' is always 'United States' — adds zero analytical value
df = df.drop("Country")
print("✓ Dropped 'Country' column (100% constant — United States)\n")

✓ Dropped 'Country' column (100% constant — United States)



### STEP 9 — Transform Columns

In [0]:
# 9a. Profit Margin % = (Profit / Sales) * 100
df = df.withColumn(
    "profit_margin_pct",
    round((col("Profit") / col("Sales")) * 100, 2)
)

In [0]:
# 9b. Loss Flag — flag orders that lost money
df = df.withColumn(
    "is_loss",
    when(col("Profit") < 0, lit("Yes")).otherwise(lit("No"))
)

In [0]:
# 9c. Discount Category — bucketed for analysis
df = df.withColumn(
    "discount_tier",
    when(col("Discount") == 0,              lit("No Discount"))
    .when(col("Discount") <= 0.2,           lit("Low (1-20%)"))
    .when(col("Discount") <= 0.5,           lit("Medium (21-50%)"))
    .otherwise(                             lit("High (>50%)"))
)

In [0]:
# 9d. Revenue After Discount (cross-check column)
df = df.withColumn(
    "effective_revenue",
    round(col("Sales") * (1 - col("Discount")), 2)
)

In [0]:
# 9e. Standardise text columns to UPPERCASE
df = df.withColumn("Category",     upper(col("Category"))) \
       .withColumn("Sub-Category", upper(col("Sub-Category"))) \
       .withColumn("Segment",      upper(col("Segment"))) \
       .withColumn("Region",       upper(col("Region")))

In [0]:
print("✓ profit_margin_pct  — (Profit / Sales) × 100")
print("✓ is_loss            — Yes/No flag for loss-making orders")
print("✓ discount_tier      — No Discount / Low / Medium / High")
print("✓ effective_revenue  — Sales × (1 - Discount)")
print("✓ Category, Sub-Category, Segment, Region → UPPERCASE")

✓ profit_margin_pct  — (Profit / Sales) × 100
✓ is_loss            — Yes/No flag for loss-making orders
✓ discount_tier      — No Discount / Low / Medium / High
✓ effective_revenue  — Sales × (1 - Discount)
✓ Category, Sub-Category, Segment, Region → UPPERCASE


In [0]:
print("\nTransformed data:")
df.limit(5).display(truncate=False)


Transformed data:


Ship Mode,Segment,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit,profit_margin_pct,is_loss,discount_tier,effective_revenue
Second Class,CONSUMER,Henderson,Kentucky,42420,SOUTH,FURNITURE,CHAIRS,731.94,3,0.0,219.582,30.0,No,No Discount,731.94
Standard Class,CONSUMER,Los Angeles,California,90032,WEST,TECHNOLOGY,PHONES,907.152,6,0.2,90.7152,10.0,No,Low (1-20%),725.72
Second Class,CONSUMER,San Francisco,California,94109,WEST,OFFICE SUPPLIES,BINDERS,22.72,4,0.2,7.384,32.5,No,Low (1-20%),18.18
Standard Class,CONSUMER,Philadelphia,Pennsylvania,19140,EAST,OFFICE SUPPLIES,ART,15.76,2,0.2,3.546,22.5,No,Low (1-20%),12.61
First Class,CORPORATE,Houston,Texas,77041,CENTRAL,OFFICE SUPPLIES,STORAGE,27.24,3,0.2,2.724,10.0,No,Low (1-20%),21.79


###STEP 10 — Aggregate Sales

In [0]:
# 10a. Category-wise performance
category_sales = df.groupBy("Category") \
    .agg(
        round(sum("Sales"),  2).alias("total_sales"),
        round(sum("Profit"), 2).alias("total_profit"),
        count("*").alias("order_count"),
        round(avg("profit_margin_pct"), 2).alias("avg_profit_margin_pct"),
        round(avg("Discount"), 4).alias("avg_discount")
    ).orderBy("total_profit", ascending=False)
 
print("Category-wise Performance:")
category_sales.display(truncate=False)

Category-wise Performance:


Category,total_sales,total_profit,order_count,avg_profit_margin_pct,avg_discount
TECHNOLOGY,836154.03,145454.95,1847,15.61,0.1323
OFFICE SUPPLIES,718735.24,122364.66,6012,13.77,0.1574
FURNITURE,741306.31,18421.81,2118,3.87,0.174


In [0]:
# 10b. Region-wise performance
region_sales = df.groupBy("Region") \
    .agg(
        round(sum("Sales"),  2).alias("total_sales"),
        round(sum("Profit"), 2).alias("total_profit"),
        count("*").alias("order_count"),
        round(avg("profit_margin_pct"), 2).alias("avg_profit_margin_pct")
    ).orderBy("total_profit", ascending=False)
 
print("Region-wise Performance:")
region_sales.display(truncate=False)

Region-wise Performance:


Region,total_sales,total_profit,order_count,avg_profit_margin_pct
WEST,725255.64,108329.81,3193,21.88
EAST,678435.2,91506.31,2845,16.71
SOUTH,391721.91,46749.43,1620,16.35
CENTRAL,500782.85,39655.88,2319,-10.38


In [0]:
# 10c. Segment-wise performance 
segment_sales = df.groupBy("Segment") \
    .agg(
        round(sum("Sales"),  2).alias("total_sales"),
        round(sum("Profit"), 2).alias("total_profit"),
        count("*").alias("order_count"),
        round(avg("profit_margin_pct"), 2).alias("avg_profit_margin_pct")
    ).orderBy("total_profit", ascending=False)
 
print("Segment-wise Performance:")
segment_sales.display(truncate=False)

Segment-wise Performance:


Segment,total_sales,total_profit,order_count,avg_profit_margin_pct
CONSUMER,1160832.77,134007.44,5183,11.16
CORPORATE,706070.13,91954.98,3015,12.15
HOME OFFICE,429292.68,60279.0,1779,14.26


In [0]:
# 10d. Sub-Category performance
subcategory_sales = df.groupBy("Category", "Sub-Category") \
    .agg(
        round(sum("Sales"),  2).alias("total_sales"),
        round(sum("Profit"), 2).alias("total_profit"),
        count("*").alias("order_count"),
        round(avg("profit_margin_pct"), 2).alias("avg_profit_margin_pct")
    ).orderBy("total_profit", ascending=False)
 
print("Sub-Category Performance (top 10 by profit):")
subcategory_sales.limit(10).display(truncate=False)
 
print("Sub-Category Performance (bottom 5 — biggest losses):")
subcategory_sales.orderBy("total_profit").limit(5).display(truncate=False)

Sub-Category Performance (top 10 by profit):


Category,Sub-Category,total_sales,total_profit,order_count,avg_profit_margin_pct
TECHNOLOGY,COPIERS,149528.03,55617.82,68,31.72
TECHNOLOGY,PHONES,330007.05,44515.73,889,11.92
TECHNOLOGY,ACCESSORIES,167380.32,41936.64,775,21.82
OFFICE SUPPLIES,PAPER,78224.14,33944.24,1359,42.56
OFFICE SUPPLIES,BINDERS,203409.17,30228.0,1522,-19.86
FURNITURE,CHAIRS,327777.76,26567.13,615,4.4
OFFICE SUPPLIES,STORAGE,223843.61,21278.83,846,8.91
OFFICE SUPPLIES,APPLIANCES,107532.16,18138.01,466,-15.69
FURNITURE,FURNISHINGS,91683.02,13052.72,956,13.69
OFFICE SUPPLIES,ENVELOPES,16476.4,6964.18,254,42.31


Sub-Category Performance (bottom 5 — biggest losses):


Category,Sub-Category,total_sales,total_profit,order_count,avg_profit_margin_pct
FURNITURE,TABLES,206965.53,-17725.48,319,-14.77
FURNITURE,BOOKCASES,114880.0,-3472.56,228,-12.66
OFFICE SUPPLIES,SUPPLIES,46673.54,-1189.1,190,11.2
OFFICE SUPPLIES,FASTENERS,3024.28,949.52,217,29.92
TECHNOLOGY,MACHINES,189238.63,3384.76,115,-7.2


In [0]:
# 10e. Ship Mode performance
shipmode_sales = df.groupBy("Ship Mode") \
    .agg(
        round(sum("Sales"),  2).alias("total_sales"),
        round(sum("Profit"), 2).alias("total_profit"),
        count("*").alias("order_count"),
        round(avg("Discount"), 4).alias("avg_discount")
    ).orderBy("total_profit", ascending=False)
 
print("Ship Mode Performance:")
shipmode_sales.display(truncate=False)

Ship Mode Performance:


Ship Mode,total_sales,total_profit,order_count,avg_discount
Standard Class,1357316.35,163969.23,5955,0.1602
Second Class,459177.05,57446.65,1943,0.1386
First Class,351380.47,48953.66,1537,0.1646
Same Day,128321.73,15871.89,542,0.1527


In [0]:
# 10f. Discount Tier Impact (KEY INSIGHT)
discount_analysis = df.groupBy("discount_tier") \
    .agg(
        count("*").alias("order_count"),
        round(sum("Sales"),  2).alias("total_sales"),
        round(sum("Profit"), 2).alias("total_profit"),
        round(avg("profit_margin_pct"), 2).alias("avg_profit_margin_pct")
    ).orderBy("avg_profit_margin_pct", ascending=False)
 
print("Discount Tier Impact on Profit:")
discount_analysis.display(truncate=False)

Discount Tier Impact on Profit:


discount_tier,order_count,total_sales,total_profit,avg_profit_margin_pct
No Discount,4787,1087277.56,320844.41,34.0
Low (1-20%),3799,846432.82,100754.78,17.42
Medium (21-50%),536,298260.04,-58804.95,-22.01
High (>50%),855,64225.17,-76552.81,-113.81


In [0]:
# 10g. Loss Orders Detail
loss_orders = df.filter(col("is_loss") == "Yes") \
    .select(
        "Segment", "Region", "Category", "Sub-Category",
        "Ship Mode", "discount_tier",
        "Sales", "Discount", "Profit", "profit_margin_pct"
    ).orderBy("Profit")
 
print(f"Loss-making orders (bottom 10 worst):")
loss_orders.limit(10).display(truncate=False)

Loss-making orders (bottom 10 worst):


Segment,Region,Category,Sub-Category,Ship Mode,discount_tier,Sales,Discount,Profit,profit_margin_pct
CONSUMER,EAST,TECHNOLOGY,MACHINES,Standard Class,High (>50%),4499.985,0.7,-6599.978,-146.67
CORPORATE,SOUTH,TECHNOLOGY,MACHINES,Same Day,Medium (21-50%),7999.98,0.5,-3839.9904,-48.0
CONSUMER,CENTRAL,OFFICE SUPPLIES,BINDERS,Standard Class,High (>50%),2177.584,0.8,-3701.8928,-170.0
HOME OFFICE,WEST,TECHNOLOGY,MACHINES,Standard Class,High (>50%),2549.985,0.7,-3399.98,-133.33
CORPORATE,CENTRAL,OFFICE SUPPLIES,BINDERS,Standard Class,High (>50%),1889.99,0.8,-2929.4845,-155.0
CONSUMER,EAST,TECHNOLOGY,MACHINES,First Class,High (>50%),1799.994,0.7,-2639.9912,-146.67
CONSUMER,CENTRAL,OFFICE SUPPLIES,BINDERS,First Class,High (>50%),1525.188,0.8,-2287.782,-150.0
CONSUMER,SOUTH,FURNITURE,TABLES,Second Class,Medium (21-50%),4297.644,0.4,-1862.3124,-43.33
CONSUMER,CENTRAL,OFFICE SUPPLIES,BINDERS,Standard Class,High (>50%),1088.792,0.8,-1850.9464,-170.0
HOME OFFICE,SOUTH,TECHNOLOGY,MACHINES,Standard Class,Medium (21-50%),22638.48,0.5,-1811.0784,-8.0


### STEP 11 — Save as Parquet

In [0]:
# Cleaned main dataframe
df.write.mode("overwrite").parquet(CLEANED_PATH)
print(f"✓ Cleaned data saved        → {CLEANED_PATH}")

✓ Cleaned data saved        → /Volumes/workspace/default/project_csv/store_sales/output/superstore_cleaned


In [0]:
# Aggregated tables
category_sales.write.mode("overwrite").parquet(CAT_PATH)
print(f"✓ Category sales saved      → {CAT_PATH}")
 
region_sales.write.mode("overwrite").parquet(REGION_PATH)
print(f"✓ Region sales saved        → {REGION_PATH}")
 
segment_sales.write.mode("overwrite").parquet(SEGMENT_PATH)
print(f"✓ Segment sales saved       → {SEGMENT_PATH}")
 
subcategory_sales.write.mode("overwrite").parquet(SUBCAT_PATH)
print(f"✓ Sub-Category sales saved  → {SUBCAT_PATH}")
 
shipmode_sales.write.mode("overwrite").parquet(SHIP_PATH)
print(f"✓ Ship Mode sales saved     → {SHIP_PATH}")
 
discount_analysis.write.mode("overwrite").parquet(DISCOUNT_PATH)
print(f"✓ Discount analysis saved   → {DISCOUNT_PATH}")
 
loss_orders.write.mode("overwrite").parquet(LOSS_PATH)
print(f"✓ Loss orders saved         → {LOSS_PATH}")

✓ Category sales saved      → /Volumes/workspace/default/project_csv/store_sales/output/category_sales
✓ Region sales saved        → /Volumes/workspace/default/project_csv/store_sales/output/region_sales
✓ Segment sales saved       → /Volumes/workspace/default/project_csv/store_sales/output/segment_sales
✓ Sub-Category sales saved  → /Volumes/workspace/default/project_csv/store_sales/output/subcategory_sales
✓ Ship Mode sales saved     → /Volumes/workspace/default/project_csv/store_sales/output/shipmode_sales
✓ Discount analysis saved   → /Volumes/workspace/default/project_csv/store_sales/output/discount_analysis
✓ Loss orders saved         → /Volumes/workspace/default/project_csv/store_sales/output/loss_orders


In [0]:
# Partitioned write — optimised for Region + Category queries
df.write.mode("overwrite") \
    .partitionBy("Region", "Category") \
    .parquet("/Volumes/workspace/default/project_csv/store_sales/output/superstore_partitioned")
print(f"✓ Partitioned parquet saved → /Volumes/workspace/default/project_csv/store_sales/output/superstore_partitioned")
print(f"  (partitioned by Region / Category)\n")

✓ Partitioned parquet saved → /Volumes/workspace/default/project_csv/store_sales/output/superstore_partitioned
  (partitioned by Region / Category)



### STEP 12 — Validate Output

In [0]:
parquet_df = spark.read.parquet(CLEANED_PATH)
parquet_count = parquet_df.count()
print(f"✓ Rows written to parquet   : {parquet_count}")
print(f"  Original raw rows         : {raw_count}")
print(f"  Rows cleaned/filtered out : {raw_count - parquet_count}")

✓ Rows written to parquet   : 9977
  Original raw rows         : 9994
  Rows cleaned/filtered out : 17


In [0]:
print("\nFinal schema of cleaned parquet:")
parquet_df.printSchema()


Final schema of cleaned parquet:
root
 |-- Ship Mode: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- profit_margin_pct: double (nullable = true)
 |-- is_loss: string (nullable = true)
 |-- discount_tier: string (nullable = true)
 |-- effective_revenue: double (nullable = true)



In [0]:
print("\nSample from cleaned parquet:")
parquet_df.limit(5).display(truncate=False)


Sample from cleaned parquet:


Ship Mode,Segment,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit,profit_margin_pct,is_loss,discount_tier,effective_revenue
Second Class,CONSUMER,Henderson,Kentucky,42420,SOUTH,FURNITURE,CHAIRS,731.94,3,0.0,219.582,30.0,No,No Discount,731.94
Standard Class,CONSUMER,Los Angeles,California,90032,WEST,TECHNOLOGY,PHONES,907.152,6,0.2,90.7152,10.0,No,Low (1-20%),725.72
Second Class,CONSUMER,San Francisco,California,94109,WEST,OFFICE SUPPLIES,BINDERS,22.72,4,0.2,7.384,32.5,No,Low (1-20%),18.18
Standard Class,CONSUMER,Philadelphia,Pennsylvania,19140,EAST,OFFICE SUPPLIES,ART,15.76,2,0.2,3.546,22.5,No,Low (1-20%),12.61
First Class,CORPORATE,Houston,Texas,77041,CENTRAL,OFFICE SUPPLIES,STORAGE,27.24,3,0.2,2.724,10.0,No,Low (1-20%),21.79


In [0]:
print("\n" + "=" * 60)
print("ETL PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 60)


ETL PIPELINE COMPLETED SUCCESSFULLY
